In [ ]:
import json
import pandas as pd
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA = 'AUDIT'

def info(msg):  print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg): print(f"ERROR:   {msg}")

s = get_active_session()

In [ ]:
#load field schmas from DOC_TYPE_CONFIG
doc_types_in_pipeline = s.sql(f"""
    SELECT DISTINCT c.DOC_TYPE
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
    WHERE c.DOC_TYPE IS NOT NULL
      AND c.DOC_TYPE != 'unknown'
""").collect()

doc_types_needed = [row['DOC_TYPE'] for row in doc_types_in_pipeline]
config_cache     = {}

info(f"Doc types pending extraction: {doc_types_needed}")

for doc_type in doc_types_needed:
    config = s.sql(f"""
        SELECT MANDATORY_FIELDS, OPTIONAL_FIELDS
        FROM {DB}.{CONFIG_SCHEMA}.DOC_TYPE_CONFIG
        WHERE DOC_TYPE = '{doc_type}'
          AND IS_ACTIVE = TRUE
    """).collect()

    if not config:
        warning(f"  No active config for '{doc_type}' - skipping")
        continue

    mandatory = json.loads(config[0]['MANDATORY_FIELDS'] or '[]')
    optional  = json.loads(config[0]['OPTIONAL_FIELDS']  or '[]')

    if not mandatory:
        warning(f"  No mandatory fields for '{doc_type}' - skipping")
        continue

    response_format = {}
    mandatory_ids   = set()

    for field in mandatory:
        fid = field.get('field_id')
        q   = field.get('question', f'What is the {fid}?')
        if fid:
            response_format[fid] = q
            mandatory_ids.add(fid)

    for field in optional:
        fid = field.get('field_id')
        q   = field.get('question', f'What is the {fid}?')
        if fid and fid not in response_format:
            response_format[fid] = q

    config_cache[doc_type] = {
        'response_format': response_format,
        'mandatory_ids':   mandatory_ids,
    }
    info(f"  [{doc_type}] {len(mandatory)} mandatory "
         f"+ {len(optional)} optional fields loaded")

if not config_cache:
    print("\nNo valid configs found — nothing to extract.")


In [ ]:
# Full Extraction

if config_cache:
    all_extracted_rows = []
    all_flat_rows = []
    all_llm_rows = [] 
    all_errors = []

    DOCUMENTS_CLASSIFIED = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED"
    DOCUMENTS_PAGES = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES"
    DOCUMENTS_EXTRACTED = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED"
    DOCUMENTS_EXTRACTED_FLAT = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED_FLAT"
    DOCUMENTS_INGESTED = f"{DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED"
    LLM_USAGE    = f"{DB}.{AUDIT_SCHEMA}.LLM_USAGE"

    # Helpers
    def estimate_tokens(text):
        return len(text) // 4 if text else 0

    def parse_extraction_result(raw):
        extracted = json.loads(raw) if isinstance(raw, str) else raw
        if not extracted:
            raise ValueError("NULL or empty response")
        if extracted.get("error"):
            raise ValueError(str(extracted["error"]))
        response = extracted.get("response") or {}
        scoring  = extracted.get("scoring") or {}
        scores   = scoring.get("scores") or {}
        if not isinstance(response, dict):
            raise ValueError(f"Unexpected response type: {type(response).__name__}")
        if not isinstance(scores, dict):
            scores = {}
        return extracted, response, scores
    
    def is_empty(value):
        if value is None:
            return True
    
        if isinstance(value, str):
            return value.strip().lower() in {"", "none", "null"}
    
        return False

    def update_parent_status(child_doc_ids, status):
        if not child_doc_ids:
            return
        escaped_ids = [str(doc_id).replace("'", "''") for doc_id in child_doc_ids]
        id_list     = ", ".join(f"'{doc_id}'" for doc_id in escaped_ids)
        s.sql(f"""
            UPDATE {DOCUMENTS_INGESTED}
            SET STATUS = '{status}'
            WHERE DOC_ID IN (
                SELECT DOC_ID
                FROM {DOCUMENTS_CLASSIFIED}
                WHERE CHILD_DOC_ID IN ({id_list})
            )
        """).collect()

    # Extract each document type
    for doc_type, cfg in config_cache.items():

        response_format_json = json.dumps(cfg["response_format"])
        mandatory_ids        = set(cfg.get("mandatory_ids", []))

        info(f"\nExtracting '{doc_type}'")

        extraction_sql = f"""
            SELECT
                c.CHILD_DOC_ID,
                c.DOC_TYPE,
                SUM(LENGTH(COALESCE(
                    p.PAGE_CONTENT_TRANSLATED,
                    p.PAGE_CONTENT
                ))) AS TOTAL_CHARS,
                AI_EXTRACT(
                    text => LISTAGG(
                        '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
                        COALESCE(
                            p.PAGE_CONTENT_TRANSLATED,
                            p.PAGE_CONTENT
                        ),
                        '\\n\\n'
                    ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER),
                    responseFormat => PARSE_JSON(?),
                    scores => TRUE
                ) AS EXTRACTED
            FROM {DOCUMENTS_CLASSIFIED} c
            INNER JOIN {DOCUMENTS_PAGES} p
                ON  p.DOC_ID      = c.DOC_ID
                AND p.PAGE_NUMBER BETWEEN c.PAGE_START AND c.PAGE_END
            LEFT JOIN {DOCUMENTS_EXTRACTED} e
                ON c.CHILD_DOC_ID = e.CHILD_DOC_ID
            WHERE c.DOC_TYPE     = ?
              AND e.CHILD_DOC_ID  IS NULL
              AND p.PAGE_CONTENT  IS NOT NULL
            GROUP BY c.CHILD_DOC_ID, c.DOC_TYPE
            ORDER BY c.CHILD_DOC_ID
        """

        try:
            results = s.sql(
                extraction_sql,
                params=[response_format_json, doc_type],
            ).collect()
            info(f"  {len(results)} document(s) returned from Cortex")

        except Exception as exc:
            error(f"  SQL extraction failed for '{doc_type}': {exc}")
            continue

        # Parse results

        for row in results:
            child_doc_id = row["CHILD_DOC_ID"]
            raw          = row["EXTRACTED"]

            try:
                extracted, response, scores = parse_extraction_result(raw)

                # Token tracking
                input_tokens  = (row["TOTAL_CHARS"] or 0) // 4
                output_tokens = estimate_tokens(json.dumps(extracted))

                all_llm_rows.append({
                    "CHILD_DOC_ID":  child_doc_id,
                    "PIPELINE_STEP": "AI_EXTRACT",
                    "MODEL":         "AI_EXTRACT",
                    "TOKENS_IN":     input_tokens,
                    "TOKENS_OUT":    output_tokens,
                })

                # Full extraction row
                all_extracted_rows.append({
                    "CHILD_DOC_ID":     child_doc_id,
                    "DOC_TYPE":         doc_type,
                    "EXTRACTED_JSON":   json.dumps(extracted),
                    "EXTRACTION_MODEL": "AI_EXTRACT",
                })

                # Flattened field rows 
                for field_id, value in response.items():
                    score_obj  = scores.get(field_id)
                    confidence = (
                        score_obj.get("score")
                        if isinstance(score_obj, dict)
                        else None
                    )
                    is_missing = is_empty(value)
                    all_flat_rows.append({
                        "CHILD_DOC_ID":     child_doc_id,
                        "FIELD_ID":         field_id,
                        "FIELD_VALUE":      str(value) if value is not None else None,
                        "FIELD_CONFIDENCE": confidence,
                        "IS_MANDATORY":     field_id in mandatory_ids,
                        "IS_MISSING":       is_missing,
                    })

                # Missing mandatory field check 
                # Treat None value same as missing 
                missing = [
                    field_id
                    for field_id in mandatory_ids
                    if is_empty(response.get(field_id))
                ]

                if missing:
                    info(
                        f"  [OK] {child_doc_id} - "
                        f"{len(response)} fields; "
                        f"missing mandatory: {missing}"
                    )
                else:
                    info(
                        f"  [OK] {child_doc_id} - "
                        f"{len(response)} fields"
                    )

            except Exception as exc:
                all_errors.append({
                    "child_doc_id": child_doc_id,
                    "doc_type":     doc_type,
                    "error":        str(exc),
                })
                error(f"  [FAIL] {child_doc_id}: {exc}")


In [ ]:
# Write DOCUMENTS_EXTRACTED

if all_extracted_rows:
    s.write_pandas(
        pd.DataFrame(all_extracted_rows),
        table_name="DOCUMENTS_EXTRACTED",
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"\nWrote {len(all_extracted_rows)} row(s) to DOCUMENTS_EXTRACTED")

#  Write DOCUMENTS_EXTRACTED_FLAT

if all_flat_rows:
    s.write_pandas(
        pd.DataFrame(all_flat_rows),
        table_name="DOCUMENTS_EXTRACTED_FLAT",
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(all_flat_rows)} row(s) to DOCUMENTS_EXTRACTED_FLAT")

In [ ]:
# Write LLM_USAGE 

if all_llm_rows:
    s.write_pandas(
        pd.DataFrame(all_llm_rows),
        table_name="LLM_USAGE",
        database=DB, schema=AUDIT_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(all_llm_rows)} row(s) to LLM_USAGE")

# Update STATUS in DOCUMENTS_INGESTED

extracted_ids = [row["CHILD_DOC_ID"] for row in all_extracted_rows]
error_ids     = [row["child_doc_id"] for row in all_errors]

if extracted_ids:
    update_parent_status(extracted_ids, "EXTRACTED")
    info(f"Updated {len(extracted_ids)} document(s) to EXTRACTED")

if error_ids:
    update_parent_status(error_ids, "EXTRACT_ERROR")
    info(f"Updated {len(error_ids)} document(s) to EXTRACT_ERROR")

In [ ]:
# Summary

print(f"\n Extraction summary")
print(f"  Extracted successfully : {len(all_extracted_rows)}")
print(f"  Fields extracted       : {len(all_flat_rows)}")
print(f"  Errors                 : {len(all_errors)}")
print(f"  Total input tokens     : {sum(r['TOKENS_IN'] for r in all_llm_rows):,}")
print(f"  Total output tokens    : {sum(r['TOKENS_OUT'] for r in all_llm_rows):,}")

if all_errors:
    print("\n  Failed documents:")
    for e in all_errors:
        print(f"    {e['child_doc_id']} ({e['doc_type']}): {e['error']}")

print(f"\n Results by doc type")
s.sql(f"""
        SELECT
            c.DOC_TYPE,
            COUNT(DISTINCT e.CHILD_DOC_ID)              AS DOCS_EXTRACTED,
            COUNT(*)                                     AS TOTAL_FIELDS,
            COUNT(CASE WHEN e.IS_MISSING = TRUE
                       AND e.IS_MANDATORY THEN 1 END)   AS MISSING_MANDATORY,
            ROUND(AVG(e.FIELD_CONFIDENCE), 3)            AS AVG_CONFIDENCE
        FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED_FLAT e
        JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
            ON e.CHILD_DOC_ID = c.CHILD_DOC_ID
        GROUP BY c.DOC_TYPE
        ORDER BY c.DOC_TYPE
""").show()

In [ ]:
%%sql -r dataframe_1
select * from permafrost_poc.processing.documents_extracted_flat

In [ ]:
%%sql -r dataframe_2
select * from permafrost_poc.processing.documents_extracted_flat where child_doc_id = '9260611b-67b4-48c3-8beb-7cc649714b4d'

In [ ]:
%%sql -r dataframe_3
select * from permafrost_poc.processing.documents_pages
where child_doc_id = '939cd95a-442f-4867-97ed-6133ff6df37c'